# AWP Skill Smoke — All 5 Autonomy Levels (A0–A4)

Generates skill-conformant workflows for **one fictional task** across every
autonomy level, validates each with `awp validate`, and runs a live E2E for
at least one representative level.

**Fictional task (red thread):** *"Lunar supply chain feasibility briefing —
assess economic, logistical, regulatory, and risk dimensions of lunar-surface
supply chain deployment and produce a structured briefing."*

Each level exercises progressively more autonomy:

| Level | Variant                          | Engine           | What it tests                                 |
|-------|----------------------------------|------------------|-----------------------------------------------|
| A0    | `lunar_brief_a0` (single agent)  | dag              | minimal viable workflow; R1–R18                |
| A1    | `lunar_brief_a1` (3-agent chain) | dag              | DAG, state sharing, multi-agent depends_on    |
| A2    | `lunar_brief_a2` (delegation)    | delegation_loop  | manager + dynamic workers + budget            |
| A3    | `lunar_brief_a3` (self-tooling)  | delegation_loop  | `dynamic_tools.enabled`, codemode.tool_creation |
| A4    | `lunar_brief_a4` (recursive)     | delegation_loop  | `max_depth`, observability, circuit_breaker   |

**Parallel-session safety:** every run uses a unique timestamp+hex suffix;
outputs live under `/tmp/awp-skill-smoke/<SESSION_ID>/` so concurrent Claude
sessions never touch each other's workspace.

In [1]:
# --- 0. Setup: paths, env, parallel-session isolation -----------------------
import json
import os
import subprocess
import sys
import uuid

import nest_asyncio
nest_asyncio.apply()  # allow asyncio.run() inside Jupyter (for _harness)
from datetime import datetime
from pathlib import Path

REPO = Path("/home/shumway/projects/agent-workflow-protocol").resolve()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "examples"))
sys.path.insert(0, str(REPO / "examples" / "e2e"))

SESSION_ID = f"skill-smoke-{datetime.now().strftime('%Y%m%d-%H%M%S')}-{uuid.uuid4().hex[:6]}"
SMOKE_ROOT = Path("/tmp/awp-skill-smoke") / SESSION_ID
SMOKE_ROOT.mkdir(parents=True, exist_ok=True)
print(f"SESSION_ID = {SESSION_ID}")
print(f"SMOKE_ROOT = {SMOKE_ROOT}")

# Load OPENROUTER_API_KEY from known location (reference memory: meta-agents/.env).
def _load_openrouter_key():
    key = os.environ.get("OPENROUTER_API_KEY")
    if key:
        return key
    env_path = Path("/home/shumway/projects/meta-agents/.env")
    if env_path.exists():
        for raw in env_path.read_text().splitlines():
            line = raw.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            k, v = line.split("=", 1)
            if k.strip() == "OPENROUTER_API_KEY":
                os.environ["OPENROUTER_API_KEY"] = v.strip().strip('"').strip("'")
                return os.environ["OPENROUTER_API_KEY"]
    return None

OPENROUTER_KEY = _load_openrouter_key()
print(f"OPENROUTER_API_KEY loaded: {bool(OPENROUTER_KEY)}")

TASK = (
    "Assess lunar supply chain feasibility: produce a briefing that covers "
    "economic viability, logistics architecture, regulatory constraints, and "
    "top operational risks for 2030-era lunar-surface supply deployments."
)
print(f"TASK: {TASK}")

SESSION_ID = skill-smoke-20260417-234715-7e0073
SMOKE_ROOT = /tmp/awp-skill-smoke/skill-smoke-20260417-234715-7e0073
OPENROUTER_API_KEY loaded: True
TASK: Assess lunar supply chain feasibility: produce a briefing that covers economic viability, logistics architecture, regulatory constraints, and top operational risks for 2030-era lunar-surface supply deployments.


In [2]:
# --- 1. Skill-conformant workflow generator --------------------------------
# Deterministic helper that writes an AWP workflow directory following
# skill/SKILL.md Phase 3 rules. Returns the workflow path.
import textwrap
import yaml

AGENT_PY_TEMPLATE = textwrap.dedent('''\
from __future__ import annotations

from pathlib import Path
from typing import Any, Optional

from awp.runtime.agent import StandaloneAgent
from awp.runtime.llm import LLMClient
from awp.runtime.tools import ToolRegistry


class Agent(StandaloneAgent):
    """{{AGENT_DESCRIPTION}}"""

    def __init__(
        self,
        agent_dir: str | Path | None = None,
        workflow_dir: str | Path | None = None,
        llm: Optional[LLMClient] = None,
        tool_registry: Optional[ToolRegistry] = None,
    ) -> None:
        super().__init__(
            agent_dir=agent_dir or Path(__file__).parent,
            workflow_dir=workflow_dir or Path(__file__).parents[2],
            llm=llm,
            tool_registry=tool_registry,
        )
''')


def write_agent(workflow_dir: Path, agent_id: str, role: str, description: str,
                fields: dict[str, dict], system_prompt: str, intro: str,
                temperature: float = 0.2, max_tokens: int = 1024,
                tools_allowed: list[str] | None = None,
                output_format: str = "json"):
    """Generate a complete agent directory (R3–R10, R17–R18)."""
    agent_dir = workflow_dir / "agents" / agent_id
    (agent_dir / "workflow" / "instructions").mkdir(parents=True, exist_ok=True)
    (agent_dir / "workflow" / "prompt").mkdir(parents=True, exist_ok=True)
    (agent_dir / "workflow" / "output_schema").mkdir(parents=True, exist_ok=True)
    (agent_dir / "workflow" / "output_schema_desc").mkdir(parents=True, exist_ok=True)

    # R17: ensure confidence field is present
    fields = dict(fields)
    fields.setdefault("confidence", {
        "type": "number", "minimum": 0.0, "maximum": 1.0,
        "description": "Confidence score (0.0-1.0)",
    })

    # agent.awp.yaml
    contract = {}
    for fname, fspec in fields.items():
        entry = {"type": fspec["type"], "description": fspec.get("description", ""), "required": True}
        if "minimum" in fspec:
            entry["minimum"] = fspec["minimum"]
        if "maximum" in fspec:
            entry["maximum"] = fspec["maximum"]
        contract[fname] = entry

    awp_yaml = {
        "awp_agent": "1.0.0",
        "identity": {"id": agent_id, "role": role, "description": description},
        "model": {"name": "", "parameters": {"temperature": temperature, "max_tokens": max_tokens}},
        "prompt": {
            "system": "workflow/instructions/SYSTEM_PROMPT.md",
            "user_template": "workflow/prompt/00_INTRO.md",
        },
        "output": {
            "format": output_format,
            "schema": "workflow/output_schema/output_schema.json",
            "contract": contract,
            "validation": {"mode": "strict", "on_invalid": "retry", "max_retries": 2},
        },
    }
    if tools_allowed:
        awp_yaml["capabilities"] = {
            "tools": {"enabled": True, "max_calls": 10, "max_parallel": 2, "timeout_per_call": 20, "allowed": tools_allowed},
        }

    (agent_dir / "agent.awp.yaml").write_text(yaml.safe_dump(awp_yaml, sort_keys=False))

    # agent.py (R3/R4)
    (agent_dir / "agent.py").write_text(
        AGENT_PY_TEMPLATE.replace("{{AGENT_DESCRIPTION}}", description)
    )

    # Prompts (R7, R8)
    (agent_dir / "workflow" / "instructions" / "SYSTEM_PROMPT.md").write_text(system_prompt)
    (agent_dir / "workflow" / "prompt" / "00_INTRO.md").write_text(intro)

    # output_schema.json (R9, R17, R18)
    schema_props = {}
    for fname, fspec in fields.items():
        entry = {"type": fspec["type"], "description": fspec.get("description", "")}
        if "minimum" in fspec: entry["minimum"] = fspec["minimum"]
        if "maximum" in fspec: entry["maximum"] = fspec["maximum"]
        schema_props[fname] = entry
    schema = {
        "$schema": "http://json-schema.org/draft-07/schema#",
        "type": "object",
        "properties": schema_props,
        "required": list(fields.keys()),
    }
    (agent_dir / "workflow" / "output_schema" / "output_schema.json").write_text(
        json.dumps(schema, indent=2)
    )

    # output_schema_desc.json (R10)
    desc = {fname: fspec.get("description", "") for fname, fspec in fields.items()}
    (agent_dir / "workflow" / "output_schema_desc" / "output_schema_desc.json").write_text(
        json.dumps(desc, indent=2)
    )
    return agent_dir

print("generator helpers ready")

generator helpers ready


In [3]:
# --- 2. A0 Prescribed: single-agent briefing summary ------------------------
def build_a0(root: Path) -> Path:
    name = "lunar_brief_a0"
    wdir = root / name
    if wdir.exists():
        import shutil; shutil.rmtree(wdir)
    wdir.mkdir(parents=True)

    workflow = {
        "awp": "1.0.0",
        "workflow": {"name": name, "version": "1.0.0", "description": "A0 briefing summary", "author": "skill-smoke", "tags": ["a0", "prescribed", "lunar"]},
        "orchestration": {
            "execution": {"mode": "sequential", "timeout": {"per_agent": 60, "total": 120}},
            "graph": [{"id": "analyst", "agent": "analyst", "depends_on": [], "share_output": ["summary", "key_dimensions"]}],
        },
        "state": {"model": "shared_dict", "sharing": {"strategy": "full"}},
    }
    (wdir / "workflow.awp.yaml").write_text(yaml.safe_dump(workflow, sort_keys=False))
    write_agent(
        wdir, "analyst", role="lunar_feasibility_analyst",
        description="Single-pass summariser for lunar supply chain briefings.",
        fields={
            "summary": {"type": "string", "description": "2-3 sentence executive summary"},
            "key_dimensions": {"type": "array", "description": "List of top dimensions considered"},
        },
        system_prompt=(
            "You are a senior space-logistics analyst. Produce a concise feasibility summary.\n\n"
            "Return strict JSON with: summary (2-3 sentences), key_dimensions (4-6 items), confidence.\n"
        ),
        intro="Task: {{task}}\n\nReturn JSON only.",
        temperature=0.2,
    )
    return wdir

A0_DIR = build_a0(SMOKE_ROOT)
print(f"[A0] generated at {A0_DIR}")
list(A0_DIR.rglob("*"))[:12]

[A0] generated at /tmp/awp-skill-smoke/skill-smoke-20260417-234715-7e0073/lunar_brief_a0


[PosixPath('/tmp/awp-skill-smoke/skill-smoke-20260417-234715-7e0073/lunar_brief_a0/workflow.awp.yaml'),
 PosixPath('/tmp/awp-skill-smoke/skill-smoke-20260417-234715-7e0073/lunar_brief_a0/agents'),
 PosixPath('/tmp/awp-skill-smoke/skill-smoke-20260417-234715-7e0073/lunar_brief_a0/agents/analyst'),
 PosixPath('/tmp/awp-skill-smoke/skill-smoke-20260417-234715-7e0073/lunar_brief_a0/agents/analyst/workflow'),
 PosixPath('/tmp/awp-skill-smoke/skill-smoke-20260417-234715-7e0073/lunar_brief_a0/agents/analyst/agent.py'),
 PosixPath('/tmp/awp-skill-smoke/skill-smoke-20260417-234715-7e0073/lunar_brief_a0/agents/analyst/agent.awp.yaml'),
 PosixPath('/tmp/awp-skill-smoke/skill-smoke-20260417-234715-7e0073/lunar_brief_a0/agents/analyst/workflow/prompt'),
 PosixPath('/tmp/awp-skill-smoke/skill-smoke-20260417-234715-7e0073/lunar_brief_a0/agents/analyst/workflow/output_schema'),
 PosixPath('/tmp/awp-skill-smoke/skill-smoke-20260417-234715-7e0073/lunar_brief_a0/agents/analyst/workflow/output_schema_desc

In [4]:
# --- 3. A1 Adaptive: planner -> researcher -> writer -------------------------
def build_a1(root: Path) -> Path:
    name = "lunar_brief_a1"
    wdir = root / name
    if wdir.exists():
        import shutil; shutil.rmtree(wdir)
    wdir.mkdir(parents=True)

    workflow = {
        "awp": "1.0.0",
        "workflow": {"name": name, "version": "1.0.0", "description": "A1 DAG briefing pipeline", "author": "skill-smoke", "tags": ["a1", "adaptive", "lunar"]},
        "orchestration": {
            "execution": {"mode": "sequential", "timeout": {"per_agent": 90, "total": 300}},
            "graph": [
                {"id": "planner", "agent": "planner", "depends_on": [], "share_output": ["research_questions", "dimensions"]},
                {"id": "researcher", "agent": "researcher", "depends_on": ["planner"], "share_output": ["findings"]},
                {"id": "writer", "agent": "writer", "depends_on": ["researcher"], "share_output": ["briefing"]},
            ],
        },
        "state": {"model": "shared_dict", "sharing": {"strategy": "selective"}},
    }
    (wdir / "workflow.awp.yaml").write_text(yaml.safe_dump(workflow, sort_keys=False))

    write_agent(
        wdir, "planner", role="research_planner",
        description="Break down the feasibility brief into concrete research questions.",
        fields={
            "research_questions": {"type": "array", "description": "4-6 concrete questions to answer"},
            "dimensions": {"type": "array", "description": "Feasibility dimensions to cover"},
        },
        system_prompt=(
            "You are a research planner. Given the task, produce 4-6 crisp research questions covering "
            "economics, logistics, regulatory, and risk. Return strict JSON: research_questions, dimensions, confidence.\n"
        ),
        intro="Task: {{task}}\n\nReturn JSON only.",
        temperature=0.2, max_tokens=512,
    )
    write_agent(
        wdir, "researcher", role="desk_researcher",
        description="Synthesize findings per research question using general knowledge.",
        fields={
            "findings": {"type": "array", "description": "One finding object per research question"},
        },
        system_prompt=(
            "You are a desk researcher. For each research question provided, produce one finding object "
            "{question, finding, evidence_level} using general knowledge. Return strict JSON: findings, confidence.\n"
        ),
        intro="Research questions:\n{{research_questions}}\n\nReturn JSON only.",
        temperature=0.3, max_tokens=1024,
    )
    write_agent(
        wdir, "writer", role="briefing_writer",
        description="Assemble a concise executive briefing from the research findings.",
        fields={
            "briefing": {"type": "string", "description": "Markdown executive briefing, ~300-500 words"},
        },
        system_prompt=(
            "You are an executive briefing writer. Turn the findings into a crisp ~300-500 word markdown "
            "briefing with the sections: Summary, Economics, Logistics, Regulatory, Risks, Recommendation. "
            "Return strict JSON: briefing, confidence.\n"
        ),
        intro="Findings:\n{{findings}}\n\nReturn JSON only.",
        temperature=0.4, max_tokens=2048,
    )
    return wdir

A1_DIR = build_a1(SMOKE_ROOT)
print(f"[A1] generated at {A1_DIR}")
sorted(p.relative_to(A1_DIR).as_posix() for p in A1_DIR.rglob("*") if p.is_file())

[A1] generated at /tmp/awp-skill-smoke/skill-smoke-20260417-234715-7e0073/lunar_brief_a1


['agents/planner/agent.awp.yaml',
 'agents/planner/agent.py',
 'agents/planner/workflow/instructions/SYSTEM_PROMPT.md',
 'agents/planner/workflow/output_schema/output_schema.json',
 'agents/planner/workflow/output_schema_desc/output_schema_desc.json',
 'agents/planner/workflow/prompt/00_INTRO.md',
 'agents/researcher/agent.awp.yaml',
 'agents/researcher/agent.py',
 'agents/researcher/workflow/instructions/SYSTEM_PROMPT.md',
 'agents/researcher/workflow/output_schema/output_schema.json',
 'agents/researcher/workflow/output_schema_desc/output_schema_desc.json',
 'agents/researcher/workflow/prompt/00_INTRO.md',
 'agents/writer/agent.awp.yaml',
 'agents/writer/agent.py',
 'agents/writer/workflow/instructions/SYSTEM_PROMPT.md',
 'agents/writer/workflow/output_schema/output_schema.json',
 'agents/writer/workflow/output_schema_desc/output_schema_desc.json',
 'agents/writer/workflow/prompt/00_INTRO.md',
 'workflow.awp.yaml']

In [5]:
# --- 4. A2 Delegating: manager + dynamic workers ----------------------------
def build_a2(root: Path) -> Path:
    name = "lunar_brief_a2"
    wdir = root / name
    if wdir.exists():
        import shutil; shutil.rmtree(wdir)
    wdir.mkdir(parents=True)

    workflow = {
        "awp": "1.0.0",
        "workflow": {"name": name, "version": "1.0.0", "description": "A2 delegation-loop briefing", "author": "skill-smoke", "tags": ["a2", "delegating", "lunar"]},
        "orchestration": {
            "engine": "delegation_loop",
            "delegation_loop": {
                "manager": "agents/manager",
                "models": {"manager": None, "worker": None},
                "budget": {
                    "max_loops": 3, "max_total_workers": 4, "max_total_tokens": 300000,
                    "max_wall_time": 240, "max_depth": 0,
                    "max_workers_per_iteration": 2, "max_rejected_completions": 2,
                },
                "worker_policy": {
                    "enforced": {"sandbox": {"type": "subprocess", "max_memory_mb": 512}, "forbidden_tools": ["shell.execute", "file.write_outside_workspace"]},
                    "manager_controlled": ["instructions", "skills", "tools_allowed", "output_contract", "temperature"],
                },
                "termination": {"enabled": True, "window": 2, "min_confidence_delta": 0.05},
                "validation": {"deterministic": {"always": True}, "llm": {"enabled": False}},
                "history": {"rolling_summary": True, "full_results_window": 2},
                "logging": {"format": "dual", "persist_artifacts": True},
            },
            "execution": {"mode": "sequential", "timeout": {"per_agent": 120, "total": 300}},
        },
        "state": {"model": "shared_dict", "sharing": {"strategy": "full"}},
    }
    (wdir / "workflow.awp.yaml").write_text(yaml.safe_dump(workflow, sort_keys=False))

    write_agent(
        wdir, "manager", role="delegation_manager",
        description="Decomposes lunar supply-chain feasibility and delegates to dynamic workers.",
        fields={
            "decision": {"type": "string", "description": "DELEGATE | COMPLETE | FAIL"},
            "rationale": {"type": "string", "description": "Why this decision"},
            "final_result": {"type": "object", "description": "Present when decision=COMPLETE"},
        },
        system_prompt=(
            "You are the delegation manager. Decompose the task into focused subtasks covering "
            "economics, logistics, regulatory, and risk. Spawn workers with crisp instructions and "
            "ephemeral skills. Emit strict JSON with fields: decision (DELEGATE|COMPLETE|FAIL), "
            "delegations[], rationale, final_result (only when COMPLETE), confidence.\n"
        ),
        intro="Task: {{task}}\n\nReturn JSON only.",
        temperature=0.2, max_tokens=2048,
    )
    return wdir

A2_DIR = build_a2(SMOKE_ROOT)
print(f"[A2] generated at {A2_DIR}")

[A2] generated at /tmp/awp-skill-smoke/skill-smoke-20260417-234715-7e0073/lunar_brief_a2


In [6]:
# --- 5. A3 Self-Tooling: delegation + dynamic_tools --------------------------
def build_a3(root: Path) -> Path:
    name = "lunar_brief_a3"
    wdir = root / name
    if wdir.exists():
        import shutil; shutil.rmtree(wdir)
    wdir.mkdir(parents=True)

    workflow = {
        "awp": "1.0.0",
        "workflow": {"name": name, "version": "1.0.0", "description": "A3 self-tooling briefing", "author": "skill-smoke", "tags": ["a3", "self-tooling", "lunar"]},
        "orchestration": {
            "engine": "delegation_loop",
            "delegation_loop": {
                "manager": "agents/manager",
                "models": {"manager": None, "worker": None},
                "budget": {
                    "max_loops": 3, "max_total_workers": 4, "max_total_tokens": 400000,
                    "max_wall_time": 300, "max_depth": 0,
                    "max_workers_per_iteration": 2, "max_rejected_completions": 2,
                },
                "worker_policy": {
                    "enforced": {
                        "sandbox": {"type": "subprocess", "max_memory_mb": 512},
                        "forbidden_tools": ["shell.execute", "file.write_outside_workspace"],
                        "codemode": {"max_tools_per_worker": 4},
                    },
                    "manager_controlled": ["instructions", "skills", "tools_allowed", "output_contract", "codemode.enabled", "codemode.tool_creation", "temperature"],
                },
                "termination": {"enabled": True, "window": 2, "min_confidence_delta": 0.05},
                "validation": {"deterministic": {"always": True}, "llm": {"enabled": False}},
                "history": {"rolling_summary": True, "full_results_window": 2},
                "logging": {"format": "dual", "persist_artifacts": True},
            },
            "execution": {"mode": "sequential", "timeout": {"per_agent": 180, "total": 600}},
        },
        "state": {"model": "shared_dict", "sharing": {"strategy": "full"}},
        "dynamic_tools": {
            "enabled": True, "persist": True, "max_total": 20,
            "allowed_namespaces": ["scoring", "analysis"],
            "code_review": True,
        },
    }
    (wdir / "workflow.awp.yaml").write_text(yaml.safe_dump(workflow, sort_keys=False))

    # Manager with codemode enabled (so tool_creation flows via the delegation envelope).
    write_agent(
        wdir, "manager", role="self_tooling_manager",
        description="Creates scoring/analysis tools at runtime, then uses them to rate feasibility.",
        fields={
            "decision": {"type": "string", "description": "DELEGATE | COMPLETE | FAIL"},
            "rationale": {"type": "string", "description": "Why this decision"},
            "final_result": {"type": "object", "description": "Present when decision=COMPLETE"},
        },
        system_prompt=(
            "You are a self-tooling delegation manager. Phase 1: spawn a tool-builder worker that creates "
            "`scoring.feasibility_score` via the `dynamic.create_tool` meta-tool (archetype=compute). Phase 2: "
            "spawn analyst workers that call this tool to rate each dimension. Return strict JSON: "
            "decision, delegations[], rationale, final_result (when COMPLETE), confidence.\n"
        ),
        intro="Task: {{task}}\n\nReturn JSON only.",
        temperature=0.2, max_tokens=2048,
    )
    # Add codemode capability block to the manager (R19-R21).
    mgr_yaml = wdir / "agents" / "manager" / "agent.awp.yaml"
    mgr = yaml.safe_load(mgr_yaml.read_text())
    mgr.setdefault("capabilities", {})
    mgr["capabilities"].setdefault("tools", {"enabled": True, "max_calls": 10, "max_parallel": 2, "timeout_per_call": 20, "allowed": ["code.execute", "file.read", "file.write"]})
    mgr["capabilities"]["codemode"] = {"enabled": True, "language": "python", "tool_creation": True, "tool_creation_namespace": "scoring"}
    mgr["capabilities"]["sandbox"] = {"type": "subprocess", "constraints": {"max_memory_mb": 512, "max_cpu_seconds": 30}}
    mgr_yaml.write_text(yaml.safe_dump(mgr, sort_keys=False))
    return wdir

A3_DIR = build_a3(SMOKE_ROOT)
print(f"[A3] generated at {A3_DIR}")

[A3] generated at /tmp/awp-skill-smoke/skill-smoke-20260417-234715-7e0073/lunar_brief_a3


In [7]:
# --- 6. A4 Self-Organizing: recursive delegation + observability ------------
def build_a4(root: Path) -> Path:
    name = "lunar_brief_a4"
    wdir = root / name
    if wdir.exists():
        import shutil; shutil.rmtree(wdir)
    wdir.mkdir(parents=True)

    workflow = {
        "awp": "1.0.0",
        "workflow": {"name": name, "version": "1.0.0", "description": "A4 recursive delegation briefing", "author": "skill-smoke", "tags": ["a4", "self-organizing", "lunar"]},
        "orchestration": {
            "engine": "delegation_loop",
            "delegation_loop": {
                "manager": "agents/manager",
                "models": {"manager": None, "worker": None},
                "budget": {
                    "max_loops": 4, "max_total_workers": 8, "max_total_tokens": 800000,
                    "max_wall_time": 420, "max_depth": 2,
                    "max_concurrent_submanagers": 2, "max_total_submanagers_per_run": 3,
                    "max_workers_per_iteration": 3, "max_rejected_completions": 2,
                },
                "worker_policy": {
                    "enforced": {
                        "sandbox": {"type": "subprocess", "max_memory_mb": 512, "max_cpu_seconds": 30, "network": False},
                        "forbidden_tools": ["shell.execute", "file.write_outside_workspace"],
                        "codemode": {"max_tools_per_worker": 4},
                        "rate_limiting": {"max_llm_calls_per_minute": 30},
                    },
                    "manager_controlled": ["instructions", "skills", "tools_allowed", "output_contract", "codemode.enabled", "codemode.tool_creation", "temperature"],
                },
                "termination": {"enabled": True, "window": 2, "min_confidence_delta": 0.05, "action": "warn_then_stop"},
                "validation": {"deterministic": {"always": True}, "llm": {"enabled": False}},
                "history": {"rolling_summary": True, "full_results_window": 2, "persist_to_disk": True},
                "logging": {"format": "dual", "persist_artifacts": True},
            },
            "execution": {"mode": "sequential", "timeout": {"per_agent": 240, "total": 900}},
        },
        "state": {"model": "shared_dict", "sharing": {"strategy": "full"}},
        "dynamic_tools": {
            "enabled": True, "persist": True, "max_total": 30,
            "allowed_namespaces": ["scoring", "analysis"], "code_review": True,
        },
        "observability": {
            "tracing": {"enabled": True, "exporter": "internal"},
            "metrics": {"enabled": True, "collector": "internal"},
            "audit": {"enabled": True, "hash_chain": True},
        },
        "security": {
            "circuit_breaker": {"enabled": True, "failure_threshold": 5, "reset_timeout": 60},
            "rate_limit": {"enabled": True, "max_calls_per_minute": 60},
        },
    }
    (wdir / "workflow.awp.yaml").write_text(yaml.safe_dump(workflow, sort_keys=False))

    write_agent(
        wdir, "manager", role="recursive_delegation_manager",
        description="Promotes complex subtasks to submanagers that own their own delegation loop.",
        fields={
            "decision": {"type": "string", "description": "DELEGATE | COMPLETE | FAIL"},
            "rationale": {"type": "string", "description": "Why this decision"},
            "final_result": {"type": "object", "description": "Present when decision=COMPLETE"},
        },
        system_prompt=(
            "You are the root recursive-delegation manager. Decompose the feasibility task into "
            "4 dimensions (economics, logistics, regulatory, risk). Promote complex dimensions to "
            "submanagers (each owning their own delegation loop within the shared budget envelope). "
            "Fold their digests back into the final briefing. Emit strict JSON: decision, delegations[], "
            "rationale, final_result (when COMPLETE), confidence.\n"
        ),
        intro="Task: {{task}}\n\nReturn JSON only.",
        temperature=0.2, max_tokens=2048,
    )
    # Add codemode capability so the manager can spawn workers that create tools.
    mgr_yaml = wdir / "agents" / "manager" / "agent.awp.yaml"
    mgr = yaml.safe_load(mgr_yaml.read_text())
    mgr.setdefault("capabilities", {})
    mgr["capabilities"].setdefault("tools", {"enabled": True, "max_calls": 10, "max_parallel": 2, "timeout_per_call": 20, "allowed": ["code.execute", "file.read", "file.write"]})
    mgr["capabilities"]["codemode"] = {"enabled": True, "language": "python", "tool_creation": True, "tool_creation_namespace": "scoring"}
    mgr["capabilities"]["sandbox"] = {"type": "subprocess", "constraints": {"max_memory_mb": 512, "max_cpu_seconds": 30}}
    mgr_yaml.write_text(yaml.safe_dump(mgr, sort_keys=False))
    return wdir

A4_DIR = build_a4(SMOKE_ROOT)
print(f"[A4] generated at {A4_DIR}")

[A4] generated at /tmp/awp-skill-smoke/skill-smoke-20260417-234715-7e0073/lunar_brief_a4


In [8]:
# --- 7. Validate all 5 levels with `awp validate` ----------------------------
LEVELS = [("A0", A0_DIR), ("A1", A1_DIR), ("A2", A2_DIR), ("A3", A3_DIR), ("A4", A4_DIR)]
validation_results = {}
for label, path in LEVELS:
    proc = subprocess.run(["awp", "validate", str(path)], capture_output=True, text=True)
    ok = proc.returncode == 0
    validation_results[label] = {"ok": ok, "stdout": proc.stdout, "stderr": proc.stderr, "rc": proc.returncode}
    marker = "✅" if ok else "❌"
    first_line = (proc.stdout + proc.stderr).splitlines()
    tail = " | ".join(first_line[-3:]) if first_line else "(no output)"
    print(f"{marker} [{label}] rc={proc.returncode}  {tail}")
validation_results

✅ [A0] rc=0  [ok] Rules passed |  | Validation passed for lunar_brief_a0


✅ [A1] rc=0  [ok] Rules passed |  | Validation passed for lunar_brief_a1


✅ [A2] rc=0   | Validation passed for lunar_brief_a2 | [FAIL] Graph must have at least one node


✅ [A3] rc=0   | Validation passed for lunar_brief_a3 | [FAIL] Graph must have at least one node


✅ [A4] rc=0   | Validation passed for lunar_brief_a4 | [FAIL] Graph must have at least one node


{'A0': {'ok': True,
  'stdout': '[ok] Manifest parsed: lunar_brief_a0 v1.0.0\n[ok] Agent parsed: analyst\n[ok] Graph valid\n[ok] Contracts valid\n[ok] Rules passed\n\nValidation passed for lunar_brief_a0\n',
  'stderr': '',
  'rc': 0},
 'A1': {'ok': True,
  'stdout': '[ok] Manifest parsed: lunar_brief_a1 v1.0.0\n[ok] Agent parsed: planner\n[ok] Agent parsed: researcher\n[ok] Agent parsed: writer\n[ok] Graph valid\n[ok] Contracts valid\n[ok] Rules passed\n\nValidation passed for lunar_brief_a1\n',
  'stderr': '',
  'rc': 0},
 'A2': {'ok': True,
  'stdout': '[ok] Manifest parsed: lunar_brief_a2 v1.0.0\n[ok] Agent parsed: manager\n[ok] Contracts valid\n[ok] Rules passed\n\nValidation passed for lunar_brief_a2\n',
  'stderr': '[FAIL] Graph must have at least one node\n',
  'rc': 0},
 'A3': {'ok': True,
  'stdout': '[ok] Manifest parsed: lunar_brief_a3 v1.0.0\n[ok] Agent parsed: manager\n[ok] Contracts valid\n[ok] Rules passed\n\nValidation passed for lunar_brief_a3\n',
  'stderr': '[FAIL] 

In [9]:
# --- 8. Show full details for any failing validation -------------------------
for label, res in validation_results.items():
    if not res["ok"]:
        print(f"===== {label} FAILED (rc={res['rc']}) =====")
        print("--- stdout ---")
        print(res["stdout"])
        print("--- stderr ---")
        print(res["stderr"])
        print()

In [10]:
# --- 9. E2E Run — A1 DAG briefing pipeline (real LLM call) -------------------
# Uses the `awp run` CLI against the real OpenRouter endpoint with a tiny
# budget. Proves the generated workflow is not just structurally valid
# but actually executes end-to-end. Set SKILL_SMOKE_SKIP_E2E=1 to bypass.
SKIP_E2E = os.environ.get("SKILL_SMOKE_SKIP_E2E") == "1"
if SKIP_E2E:
    print("⚠  SKILL_SMOKE_SKIP_E2E=1 — skipping A1 `awp run`")
elif not OPENROUTER_KEY:
    print("⚠  Skipping E2E: OPENROUTER_API_KEY not available")
else:
    cmd = [
        "awp", "run", str(A1_DIR),
        "--task", TASK,
        "--model", "openai/gpt-5-mini",
        "--debug",
    ]
    print("$ " + " ".join(cmd))
    env = os.environ.copy()
    env["LLM_MODEL"] = "openai/gpt-5-mini"
    env["AWP_NO_WIZARD"] = "1"
    proc = subprocess.run(cmd, capture_output=True, text=True, env=env, timeout=900)
    print(f"rc={proc.returncode}")
    print("--- stdout (tail 60) ---")
    print("\n".join(proc.stdout.splitlines()[-60:]))
    if proc.returncode != 0:
        print("--- stderr (tail 40) ---")
        print("\n".join(proc.stderr.splitlines()[-40:]))


$ awp run /tmp/awp-skill-smoke/skill-smoke-20260417-234715-7e0073/lunar_brief_a1 --task Assess lunar supply chain feasibility: produce a briefing that covers economic viability, logistics architecture, regulatory constraints, and top operational risks for 2030-era lunar-surface supply deployments. --model openai/gpt-5-mini --debug


rc=0
--- stdout (tail 60) ---
      
      - Current framework: Outer Space Treaty governs state responsibility and non-appropriation; national laws (e.g., U.S. commercial space statutes, Luxembourg) provide some property/extraction clarity but international consensus is limited.
      - Compliance and licensing: launch/service providers must navigate export controls (ITAR/EAR), national launch licensing, spectrum allocation, and planetary protection guidelines (COSPAR).
      - Gap areas: no comprehensive international regulatory regime for resource commercialization, liability allocation for commercial surface operations, or traffic management in cislunar space.
      
      # Risks
      
      - Technical: launch/lander failures, dust abrasion on systems, power/thermal extremes, and ISRU underperformance.
      - Operational: single-point logistics failures, comms blackouts, and resupply schedule slips.
      - Financial & political: funding discontinuity, uncertain demand, and cro

In [11]:
# --- 10. Programmatic A2 delegation-loop E2E (AgentWorkflow API) -------------
# Tiny budget — verifies the delegation loop actually spawns workers against
# real LLM responses. Experiment appears live in the AWP UI sidebar.
a2_result = None
if SKIP_E2E:
    print("⚠  SKILL_SMOKE_SKIP_E2E=1 — skipping A2 delegation-loop E2E")
elif OPENROUTER_KEY:
    from _harness import run_e2e  # lives under examples/e2e/
    a2_result = run_e2e(
        slug="skill-smoke-a2",
        title=f"Skill smoke A2 {SESSION_ID}",
        task=TASK + " Keep it short: one paragraph per dimension. IMPORTANT: the worker MUST write the final briefing to _output_dir + \"/briefing.json\" as a JSON object with keys {economic_viability, logistics_architecture, regulatory_constraints, operational_risks, confidence}.",
        model="openai/gpt-5-mini",
        worker_model="openai/gpt-5-mini",
        max_loops=3,
        max_total_workers=5,
        max_total_tokens=200_000,
        max_wall_time=420,
        max_depth=0,
        tags=["skill-smoke", "a2", "lunar"],
        extra_config={"critique": {"enabled": False}},
    )
    print(json.dumps({"status": a2_result.get("status"), "slug": a2_result.get("slug"), "workflow_dir": a2_result.get("workflow_dir")}, indent=2))
else:
    print("⚠  Skipping A2 E2E: OPENROUTER_API_KEY not available")


DEBUG:awp.runtime.secrets:Loaded 16 entries from global /home/shumway/.awp/.env


INFO:awp.data.workflow:Preparing inputs in workspace: /tmp/awp-experiments/skill-smoke-a2-20260417-234802-a74ea0


INFO:awp.data.workflow:Starting delegation loop: task=Assess lunar supply chain feasibility: produce a briefing that covers economic v


INFO:awp.runtime.delegation_loop_runner:Decision journal active (max_entries=20)


INFO:awp.runtime.delegation_loop_runner:Task planning active (max_subtasks=10)


...

DEBUG:aiosqlite:executing functools.partial(<built-in method close of sqlite3.Connection object at 0x702c2afa9c60>)


DEBUG:aiosqlite:operation functools.partial(<built-in method close of sqlite3.Connection object at 0x702c2afa9c60>) completed


DEBUG:aiosqlite:executing <function Connection.stop.<locals>.close_and_stop at 0x702c2afa2340>


DEBUG:aiosqlite:operation <function Connection.stop.<locals>.close_and_stop at 0x702c2afa2340> completed



  AWP DELEGATION LOOP DEBUG REPORT
  Model:         openai/gpt-5-mini
  Worker model:  openai/gpt-5-mini
  Budget:        loops=3, workers=5, tokens=200,000, wall_time=420s, depth=0

  ────────────────────────────────────────────────────────
  Iteration 001
  ────────────────────────────────────────────────────────
    ────────────────────────────────────────────────
    MANAGER DECISION
    ────────────────────────────────────────────────
    Decision:    plan
    Reasoning:
      | Break the single deliverable (a short, four-paragraph briefing saved as briefing.json) into a small, focused pipeline so a worker can deterministically generate, validate, and write the JSON output to _output_dir. We'll have one subtask that produces the content and writes the required JSON file; its tool manifest uses archetypes to compute the text and render the JSON file. This keeps the loop simple (one worker will run in codemode and produce the required file) while satisfying R31 (every subtask lists

In [12]:
# --- 11. Summary -------------------------------------------------------------
print("=" * 72)
print(f"AWP Skill Smoke — session {SESSION_ID}")
print("=" * 72)
print(f"  Output root:          {SMOKE_ROOT}")
print()
for label, res in validation_results.items():
    mark = "✅" if res["ok"] else "❌"
    print(f"  [{label}] awp validate: {mark} rc={res['rc']}")
print()
print("  E2E runs:")
print("    - A1 via `awp run` (see cell 9 output)")
print(f"    - A2 via AgentWorkflow: {a2_result.get('status') if a2_result else 'skipped'}")
print()
print("Every generated workflow lives under SMOKE_ROOT. Cleanup (when done):")
print(f"    rm -rf {SMOKE_ROOT}")

AWP Skill Smoke — session skill-smoke-20260417-234715-7e0073
  Output root:          /tmp/awp-skill-smoke/skill-smoke-20260417-234715-7e0073

  [A0] awp validate: ✅ rc=0
  [A1] awp validate: ✅ rc=0
  [A2] awp validate: ✅ rc=0
  [A3] awp validate: ✅ rc=0
  [A4] awp validate: ✅ rc=0

  E2E runs:
    - A1 via `awp run` (see cell 9 output)
    - A2 via AgentWorkflow: complete

Every generated workflow lives under SMOKE_ROOT. Cleanup (when done):
    rm -rf /tmp/awp-skill-smoke/skill-smoke-20260417-234715-7e0073
